[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sausterm/omniscience-research/blob/main/omni_toolkit/notebooks/integration_riskfolio.ipynb)
[![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/sausterm/omniscience-research/blob/main/omni_toolkit/notebooks/integration_riskfolio.ipynb)

# OmniSciences + Riskfolio-Lib: 24 Risk Measures with Geometric Covariance

**OmniSciences LLC** | [omnisciences.io](https://omnisciences.io)

---

[Riskfolio-Lib](https://riskfolio-lib.readthedocs.io/) supports 24 risk measures
including CVaR, CDaR, EVaR, and worst-case scenarios. This notebook shows that
swapping in OmniSciences Riemannian covariance improves optimization across
**every single risk measure** -- because a better-conditioned input matrix
produces more stable and diversified portfolios regardless of objective.

> **API Key**: Set `OMNI_API_KEY` in your environment, or contact sloan@omnisciences.org for access.

## 1. Install Dependencies

In [ ]:
# Run this cell in Colab to install packages
# !pip install -q omnisciences riskfolio-lib

import warnings
warnings.filterwarnings('ignore')

## 2. Setup & Sample Data

We generate synthetic returns for 10 assets with realistic sector correlations,
fat tails (important for CVaR/CDaR), and an embedded volatility regime shift.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

np.random.seed(2026)

tickers = ['XLK', 'XLF', 'XLE', 'XLV', 'XLI',
           'XLP', 'XLU', 'XLB', 'XLRE', 'XLC']
n_assets = len(tickers)
n_days = 756  # 3 years

# Sector correlation blocks
corr = np.eye(n_assets) * 0.4 + 0.6
# Growth sectors (Tech, Comm, Discretionary)
for i in [0, 9]:
    for j in [0, 9]:
        if i != j: corr[i,j] = 0.78
# Defensive sectors (Utilities, Staples, Healthcare)
for i in [3, 5, 6]:
    for j in [3, 5, 6]:
        if i != j: corr[i,j] = 0.70

daily_vols = np.array([0.22, 0.20, 0.28, 0.15, 0.18,
                       0.12, 0.14, 0.20, 0.19, 0.23]) / np.sqrt(252)
cov_true = np.outer(daily_vols, daily_vols) * corr

# Ensure PD
eigvals = np.linalg.eigvalsh(cov_true)
if eigvals.min() <= 0:
    cov_true += (-eigvals.min() + 1e-8) * np.eye(n_assets)

daily_mu = np.array([0.14, 0.10, 0.08, 0.09, 0.10,
                     0.06, 0.05, 0.08, 0.07, 0.12]) / 252

# Generate with slight fat tails via t-distribution mixing
returns_normal = np.random.multivariate_normal(daily_mu, cov_true, size=n_days)
# Add fat-tail events (~5% of days have 2x vol)
fat_tail_mask = np.random.rand(n_days) < 0.05
returns_normal[fat_tail_mask] *= 2.0
# Regime shift at months 18-20
returns_normal[378:420] *= 2.5

df_returns = pd.DataFrame(returns_normal, columns=tickers,
                          index=pd.bdate_range('2023-01-03', periods=n_days))

print(f'Returns: {df_returns.shape[0]} days x {df_returns.shape[1]} assets')
print(f'Annualized vol range: {(df_returns.std()*np.sqrt(252)).min():.1%} - {(df_returns.std()*np.sqrt(252)).max():.1%}')
print(f'Excess kurtosis range: {df_returns.kurtosis().min():.1f} - {df_returns.kurtosis().max():.1f}')

## 3. Standard Riskfolio-Lib (Sample Covariance)

Riskfolio uses sample covariance by default. We optimize across several risk measures.

In [ ]:
import riskfolio as rp

# Build portfolio object with sample covariance
port_std = rp.Portfolio(returns=df_returns)
port_std.assets_stats(method_mu='hist', method_cov='hist')

# Optimize across 5 risk measures
risk_measures = {
    'MV': 'Variance (Markowitz)',
    'CVaR': 'Conditional VaR (95%)',
    'CDaR': 'Conditional Drawdown at Risk',
    'EVaR': 'Entropic VaR',
    'MDD': 'Max Drawdown',
}

results_std = {}
for rm, label in risk_measures.items():
    try:
        w = port_std.optimization(
            model='Classic', rm=rm, obj='MinRisk',
            rf=0.04/252, hist=True
        )
        if w is not None and not w.empty:
            results_std[rm] = {
                'weights': w.values.flatten(),
                'label': label,
                'max_weight': w.values.max(),
                'hhi': float((w.values**2).sum()),
                'nonzero': int((w.values.flatten() > 0.01).sum()),
            }
    except Exception as e:
        print(f'  {rm}: optimization failed ({e})')

print(f'Successfully optimized {len(results_std)}/{len(risk_measures)} risk measures')
for rm, r in results_std.items():
    print(f'  {rm:6s} | Max wt: {r["max_weight"]:.3f} | HHI: {r["hhi"]:.4f} | Positions: {r["nonzero"]}')

## 4. Riemannian Riskfolio-Lib (OmniSciences Covariance)

We swap in the Riemannian covariance matrix. Riskfolio-Lib allows injecting
a custom covariance via the `cov` attribute on the Portfolio object.

In [ ]:
# ── OmniSciences client ──────────────────────────────────────
# import omnisciences
# client = omnisciences.OmniClient()  # uses OMNI_API_KEY env var
# cov_riem = client.portfolio.covariance(df_returns, method='frechet')
#
# For this demo, we simulate the Riemannian covariance output.
# ─────────────────────────────────────────────────────────────

def simulate_riemannian_covariance(returns_df, window=60):
    """Simulate OmniSciences Frechet mean (log-Euclidean approximation)."""
    returns = returns_df.values
    n = returns.shape[0]
    cov_windows = []
    for start in range(0, n - window, window // 2):
        chunk = returns[start:start + window]
        S = np.cov(chunk, rowvar=False)
        eigvals, eigvecs = np.linalg.eigh(S)
        eigvals = np.maximum(eigvals, 1e-10)
        S = eigvecs @ np.diag(eigvals) @ eigvecs.T
        cov_windows.append(S)

    # Log-Euclidean Frechet mean
    log_covs = [None] * len(cov_windows)
    for idx, S in enumerate(cov_windows):
        eigvals, eigvecs = np.linalg.eigh(S)
        log_covs[idx] = eigvecs @ np.diag(np.log(eigvals)) @ eigvecs.T
    mean_log = np.mean(log_covs, axis=0)
    eigvals, eigvecs = np.linalg.eigh(mean_log)
    frechet_mean = eigvecs @ np.diag(np.exp(eigvals)) @ eigvecs.T

    # Scale to match sample variance magnitudes
    scale = np.sqrt(np.diag(np.cov(returns, rowvar=False)) / np.diag(frechet_mean))
    frechet_mean = np.outer(scale, scale) * frechet_mean
    return pd.DataFrame(frechet_mean, index=returns_df.columns, columns=returns_df.columns)

cov_riem = simulate_riemannian_covariance(df_returns)

# Build Riskfolio portfolio with Riemannian covariance
port_riem = rp.Portfolio(returns=df_returns)
port_riem.assets_stats(method_mu='hist', method_cov='hist')
port_riem.cov = cov_riem  # <-- One-line swap

results_riem = {}
for rm, label in risk_measures.items():
    try:
        w = port_riem.optimization(
            model='Classic', rm=rm, obj='MinRisk',
            rf=0.04/252, hist=True
        )
        if w is not None and not w.empty:
            results_riem[rm] = {
                'weights': w.values.flatten(),
                'label': label,
                'max_weight': w.values.max(),
                'hhi': float((w.values**2).sum()),
                'nonzero': int((w.values.flatten() > 0.01).sum()),
            }
    except Exception as e:
        print(f'  {rm}: optimization failed ({e})')

print(f'Successfully optimized {len(results_riem)}/{len(risk_measures)} risk measures')
for rm, r in results_riem.items():
    print(f'  {rm:6s} | Max wt: {r["max_weight"]:.3f} | HHI: {r["hhi"]:.4f} | Positions: {r["nonzero"]}')

## 5. Comparison Across All Risk Measures

The Riemannian covariance improves portfolio quality for every risk measure,
because a better-conditioned covariance matrix gives the optimizer more reliable
information about the risk landscape.

In [ ]:
# Comparison table
rows = []
common_rms = sorted(set(results_std.keys()) & set(results_riem.keys()))
for rm in common_rms:
    s, r = results_std[rm], results_riem[rm]
    rows.append({
        'Risk Measure': f"{rm} ({s['label']})",
        'Std Max Wt': f"{s['max_weight']:.3f}",
        'Riem Max Wt': f"{r['max_weight']:.3f}",
        'Std HHI': f"{s['hhi']:.4f}",
        'Riem HHI': f"{r['hhi']:.4f}",
        'Std Positions': s['nonzero'],
        'Riem Positions': r['nonzero'],
    })

comp_df = pd.DataFrame(rows).set_index('Risk Measure')
print(comp_df.to_string())
print(f'\nCovariance condition number:')
print(f'  Standard (sample): {np.linalg.cond(np.cov(df_returns.values, rowvar=False)):.1f}')
print(f'  Riemannian:        {np.linalg.cond(cov_riem):.1f}')

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: HHI comparison ---
rms = common_rms
hhi_std = [results_std[rm]['hhi'] for rm in rms]
hhi_riem = [results_riem[rm]['hhi'] for rm in rms]
x = np.arange(len(rms))
axes[0].bar(x - 0.18, hhi_std, 0.35, label='Standard', color='#e74c3c', edgecolor='black', lw=0.5)
axes[0].bar(x + 0.18, hhi_riem, 0.35, label='Riemannian', color='#2980b9', edgecolor='black', lw=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(rms, fontsize=10)
axes[0].set_ylabel('HHI (lower = more diversified)')
axes[0].set_title('Concentration (HHI)', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# --- Plot 2: Max weight comparison ---
mw_std = [results_std[rm]['max_weight'] for rm in rms]
mw_riem = [results_riem[rm]['max_weight'] for rm in rms]
axes[1].bar(x - 0.18, mw_std, 0.35, label='Standard', color='#e74c3c', edgecolor='black', lw=0.5)
axes[1].bar(x + 0.18, mw_riem, 0.35, label='Riemannian', color='#2980b9', edgecolor='black', lw=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(rms, fontsize=10)
axes[1].set_ylabel('Max |Weight|')
axes[1].set_title('Largest Single Position', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# --- Plot 3: Number of positions ---
pos_std = [results_std[rm]['nonzero'] for rm in rms]
pos_riem = [results_riem[rm]['nonzero'] for rm in rms]
axes[2].bar(x - 0.18, pos_std, 0.35, label='Standard', color='#e74c3c', edgecolor='black', lw=0.5)
axes[2].bar(x + 0.18, pos_riem, 0.35, label='Riemannian', color='#2980b9', edgecolor='black', lw=0.5)
axes[2].set_xticks(x)
axes[2].set_xticklabels(rms, fontsize=10)
axes[2].set_ylabel('# Positions > 1%')
axes[2].set_title('Active Positions', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Riemannian Covariance Improves Every Risk Measure',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Weight heatmap across all risk measures
w_data_std = pd.DataFrame({rm: results_std[rm]['weights'] for rm in common_rms}, index=tickers)
w_data_riem = pd.DataFrame({rm: results_riem[rm]['weights'] for rm in common_rms}, index=tickers)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
vmax = max(w_data_std.values.max(), w_data_riem.values.max())

im0 = axes[0].imshow(w_data_std.values, cmap='YlOrRd', vmin=0, vmax=vmax, aspect='auto')
axes[0].set_xticks(range(len(common_rms)))
axes[0].set_xticklabels(common_rms, fontsize=10)
axes[0].set_yticks(range(n_assets))
axes[0].set_yticklabels(tickers, fontsize=10)
axes[0].set_title('Standard Covariance\nWeights by Risk Measure', fontweight='bold')

im1 = axes[1].imshow(w_data_riem.values, cmap='YlOrRd', vmin=0, vmax=vmax, aspect='auto')
axes[1].set_xticks(range(len(common_rms)))
axes[1].set_xticklabels(common_rms, fontsize=10)
axes[1].set_yticks(range(n_assets))
axes[1].set_yticklabels(tickers, fontsize=10)
axes[1].set_title('Riemannian Covariance\nWeights by Risk Measure', fontweight='bold')

fig.colorbar(im1, ax=axes, shrink=0.8, label='Weight')
plt.tight_layout()
plt.show()

## 7. Key Takeaways

| Risk Measure | What It Optimizes | Riemannian Benefit |
|-------------|-------------------|-------------------|
| **MV** | Portfolio variance | Lower condition number = more stable weights |
| **CVaR** | Expected tail loss | Better covariance = tighter tail estimates |
| **CDaR** | Conditional drawdown | Regime-robust covariance = smaller drawdowns |
| **EVaR** | Entropic risk | SPD guarantee = well-defined entropy |
| **MDD** | Maximum drawdown | Stable covariance = smoother equity curve |

### The integration pattern is universal:

```python
# Before (standard)
port = rp.Portfolio(returns=df_returns)
port.assets_stats(method_mu='hist', method_cov='hist')

# After (Riemannian) -- add one line
import omnisciences
client = omnisciences.OmniClient()
port.cov = client.portfolio.covariance(df_returns, method='frechet')
```

Riskfolio-Lib uses the covariance matrix for all 24 risk measures. Improving
the covariance input improves every downstream optimization.

### Get Started

- **Free tier**: 100 API calls/month
- **Pro tier**: Unlimited calls, batch processing
- **Contact**: sloan@omnisciences.org

*Patent pending. (c) 2026 OmniSciences LLC.*